# FreshCheck Swin-T Structured Multi-View Runner

Notebook ???????????????????? `Swin-T + structured multi-view` ????????? multi-view ??? Kwon

????????????????:
- ?????? random patch ????
- ??? 3 ?????????????????????????????
- ?????????????????????: `top / center / bottom`
- ?????????????????????: `left / center / right`
- train ?? jitter ??????????? anchor ??? eval ???? deterministic

???????? runtime:
- `Python 3`
- `A100 GPU`
- `High-RAM: ON`


In [ ]:
#@title 1) Clone / Update repository
REPO_URL = "https://github.com/techasit239/DADS7202_PigPicture.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/DADS7202_PigPicture"  #@param {type:"string"}

import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

%cd /content/DADS7202_PigPicture


In [ ]:
#@title 2) Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')
ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts'
EXPERIMENTS_ROOT = ARTIFACTS_ROOT / 'experiments_manual'
EXPERIMENTS_AUTO_ROOT = ARTIFACTS_ROOT / 'experiments_auto'
TARGET_SPLIT_ROOT = ARTIFACTS_ROOT / 'target_split'

print('Drive root:', DRIVE_ROOT)
print('Artifacts :', ARTIFACTS_ROOT)


In [ ]:
#@title 3) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt timm transformers


In [ ]:
#@title 4) Verify CLI supports structured multi-view
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py train --help | head -n 140

print()
print('Expected flags:')
print('--sampling-mode')
print('--num-patches')
print('--freeze-backbone-epochs')
print('--amp')
print('--prefetch-factor')


In [ ]:
#@title 5) Runtime check
import torch

print('cuda available :', torch.cuda.is_available())
print('device count    :', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device name     :', torch.cuda.get_device_name(0))
    print('capability      :', torch.cuda.get_device_capability(0))

assert torch.cuda.is_available(), 'GPU not found. Please switch runtime to A100 GPU.'


In [ ]:
#@title 6) Experiment paths
TRAIN_CSV = str(EXPERIMENTS_ROOT / 'exp_c_train.csv')
VAL_CSV = str(EXPERIMENTS_ROOT / 'exp_c_val.csv')
TARGET_CSV = str(TARGET_SPLIT_ROOT / 'target_holdout.csv')
SOURCE_CSV = str(EXPERIMENTS_AUTO_ROOT / 'source_holdout.csv')
MULTIVIEW_ROOT = str(ARTIFACTS_ROOT / 'exp_swin_multiview_runs')

for p in [TRAIN_CSV, VAL_CSV, TARGET_CSV, SOURCE_CSV]:
    print(p)
print('Output root:', MULTIVIEW_ROOT)


In [ ]:
#@title 7) Global settings
SEEDS = "42 52 62 72 82"  #@param {type:"string"}
IMG_SIZE = 224  #@param {type:"integer"}
NUM_VIEWS = 3  #@param {type:"integer"}
NUM_WORKERS = 4  #@param {type:"integer"}
PREFETCH_FACTOR = 4  #@param {type:"integer"}
AMP_DTYPE = "bf16"  #@param ["bf16", "fp16"]
RUN_SOURCE_EVAL = True  #@param {type:"boolean"}

SEED_LIST = [int(x) for x in SEEDS.split()]
print('Seeds:', SEED_LIST)
print('Views:', NUM_VIEWS)


In [ ]:
#@title 8) Train settings for Swin-T structured multi-view
EPOCHS = 14  #@param {type:"integer"}
PATIENCE = 5  #@param {type:"integer"}
LR = 0.0002  #@param {type:"number"}
WEIGHT_DECAY = 0.01  #@param {type:"number"}
DROPOUT = 0.30  #@param {type:"number"}
BATCH_SIZE = 16  #@param {type:"integer"}
FREEZE_BACKBONE_EPOCHS = 2  #@param {type:"integer"}

print({
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'batch_size': BATCH_SIZE,
    'freeze_backbone_epochs': FREEZE_BACKBONE_EPOCHS,
})


In [ ]:
#@title 9) Helper to run shell commands
import subprocess

def run_cmd(cmd: str):
    print(cmd)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')


## 10) Run 5-seed experiment

Cell ?????:
- train `swin_t`
- ??? `--sampling-mode structured`
- ??? 3 views ??????
- evaluate ???? `target holdout`
- ??? `source holdout` ??????? `RUN_SOURCE_EVAL = True`


In [ ]:
#@title 10) Train + eval Swin-T structured multi-view
RUNNER_PY = "/content/DADS7202_PigPicture/run_freshcheck.py"

for seed in SEED_LIST:
    out = f"{MULTIVIEW_ROOT}/swin_t/seed_{seed}"

    train_cmd = f'''python {RUNNER_PY} train \
  --train-csv "{TRAIN_CSV}" \
  --val-csv "{VAL_CSV}" \
  --output-dir "{out}/train" \
  --models swin_t \
  --epochs {EPOCHS} \
  --patience {PATIENCE} \
  --lr {LR} \
  --weight-decay {WEIGHT_DECAY} \
  --dropout {DROPOUT} \
  --img-size {IMG_SIZE} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --sampling-mode structured \
  --num-patches {NUM_VIEWS} \
  --freeze-backbone-epochs {FREEZE_BACKBONE_EPOCHS} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(train_cmd)

    target_cmd = f'''python {RUNNER_PY} evaluate \
  --csv "{TARGET_CSV}" \
  --output-dir "{out}/eval_target" \
  --models swin_t \
  --checkpoint-paths swin_t="{out}/train/checkpoints/swin_t_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --sampling-mode structured \
  --num-patches {NUM_VIEWS} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(target_cmd)

    if RUN_SOURCE_EVAL:
        source_cmd = f'''python {RUNNER_PY} evaluate \
  --csv "{SOURCE_CSV}" \
  --output-dir "{out}/eval_source" \
  --models swin_t \
  --checkpoint-paths swin_t="{out}/train/checkpoints/swin_t_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --sampling-mode structured \
  --num-patches {NUM_VIEWS} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
        run_cmd(source_cmd)


In [ ]:
#@title 11) Summarize target-holdout results across seeds
SUMMARY_PY = "/content/DADS7202_PigPicture/summarize_seed_runs.py"
SUMMARY_ROOT = f"{MULTIVIEW_ROOT}/_summary"

run_cmd(f'''python {SUMMARY_PY} \
  --root-dir "{MULTIVIEW_ROOT}/swin_t" \
  --model swin_t \
  --eval-subdir eval_target \
  --output-dir "{SUMMARY_ROOT}/swin_t_target"''')


In [ ]:
#@title 12) Summarize source-holdout results across seeds
if RUN_SOURCE_EVAL:
    run_cmd(f'''python {SUMMARY_PY} \
  --root-dir "{MULTIVIEW_ROOT}/swin_t" \
  --model swin_t \
  --eval-subdir eval_source \
  --output-dir "{SUMMARY_ROOT}/swin_t_source"''')
else:
    print('RUN_SOURCE_EVAL is False, skipped.')


In [ ]:
#@title 13) View target summary table
import pandas as pd
from pathlib import Path

summary_csv = Path(SUMMARY_ROOT) / 'swin_t_target' / 'swin_t_seed_runs.csv'
summary_json = Path(SUMMARY_ROOT) / 'swin_t_target' / 'swin_t_seed_summary.json'

print('summary csv :', summary_csv)
print('summary json:', summary_json)

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
else:
    print('Run cell 11 first.')


In [ ]:
#@title 14) Show one confusion matrix image
from IPython.display import Image, display
from pathlib import Path

SEED_TO_PREVIEW = SEED_LIST[0]  #@param {type:"raw"}
cm_path = Path(MULTIVIEW_ROOT) / 'swin_t' / f'seed_{SEED_TO_PREVIEW}' / 'eval_target' / 'swin_t_confusion_matrix.png'
print(cm_path)
if cm_path.exists():
    display(Image(filename=str(cm_path)))
else:
    print('Confusion matrix not found yet. Run cell 10 first.')
